# Matrix Multiplication : Sequential vs Multiprocessing

Matrix size = 500 x 500

**Note :** On Windows, multiprocessing uses `spawn`, so the worker function
cannot be defined inside a notebook cell (it cannot be pickled).
So the functions are kept in `matmul.py` and imported here.


In [ ]:
import time
from multiprocessing import Pool, cpu_count
from matmul import make_matrix, multiply_rows

N = 500
workers = cpu_count()

print('Matrix size :', N, 'x', N)
print('CPU cores   :', workers)


### Create the two random matrices


In [ ]:
A = make_matrix(N)
B = make_matrix(N)
print('matrices created')


### 1. Sequential method

All 500 rows are given to a single process.


In [ ]:
t1 = time.time()
C1 = multiply_rows((A, B))
seq_time = time.time() - t1

print('Sequential time : %.2f sec' % seq_time)


### 2. Multiprocessing method

The rows of A are divided into equal parts and each part is given to one process.


In [ ]:
def parallel(A, B, workers):
    chunk = len(A) // workers
    tasks = []
    for p in range(workers):
        start = p * chunk
        end = len(A) if p == workers - 1 else start + chunk
        tasks.append((A[start:end], B))

    with Pool(workers) as pool:
        parts = pool.map(multiply_rows, tasks)

    # join the row blocks back into one matrix
    C = []
    for part in parts:
        C += part
    return C


In [ ]:
t2 = time.time()
C2 = parallel(A, B, workers)
par_time = time.time() - t2

print('Multiprocessing time : %.2f sec' % par_time)


### 3. Result


In [ ]:
print('Sequential time      : %.2f sec' % seq_time)
print('Multiprocessing time : %.2f sec' % par_time)
print('Speedup              : %.2fx' % (seq_time / par_time))
print('Efficiency           : %.2f%%' % (seq_time / par_time / workers * 100))

# both methods must give the same answer
assert C1 == C2, 'results do not match!'
print()
print('Result verified : both matrices are same')


### Conclusion

Sequential execution time is much higher than multiprocessing, because
multiprocessing divides the rows among all the CPU cores and computes them
at the same time.

The speedup is less than the number of cores because Windows has to start a
new Python process for every worker and copy matrix B to each of them. This
extra work is called parallel overhead.
